# 🔥 Chapter 2 — Tensors: The Foundation of Everything

---

### 🎯 What you'll learn
- Creating tensors (from data, random, zeros, ones, ranges)
- Tensor attributes: shape, dtype, device
- Indexing, slicing, and advanced indexing
- Reshaping, stacking, concatenating
- ⚠️ Critical corner cases (views vs copies, contiguous memory)

## 2.1 — Creating Tensors

In [ ]:
import torch

# From Python data
t1 = torch.tensor([1, 2, 3])                      # 1D
t2 = torch.tensor([[1, 2], [3, 4]])                # 2D
t3 = torch.tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])  # 3D

# Special tensors
zeros = torch.zeros(3, 4)         # 3x4 of zeros
ones = torch.ones(2, 3)           # 2x3 of ones
eye = torch.eye(3)                # 3x3 identity matrix
full = torch.full((2, 3), 7.0)    # 2x3 filled with 7.0

# Random tensors
rand = torch.rand(3, 3)           # Uniform [0, 1)
randn = torch.randn(3, 3)         # Normal (mean=0, std=1)
randint = torch.randint(0, 10, (3, 3))  # Random integers [0, 10)

# Sequences
arange = torch.arange(0, 10, 2)          # [0, 2, 4, 6, 8]
linspace = torch.linspace(0, 1, 5)       # [0.0, 0.25, 0.5, 0.75, 1.0]

print("zeros:\n", zeros)
print("randn:\n", randn)
print("arange:", arange)
print("linspace:", linspace)

## 2.2 — Tensor Attributes

In [ ]:
t = torch.randn(3, 4, 5)

print(f"Shape:       {t.shape}")        # torch.Size([3, 4, 5])
print(f"Dimensions:  {t.ndim}")          # 3
print(f"Dtype:       {t.dtype}")         # torch.float32
print(f"Device:      {t.device}")        # cpu
print(f"Total elems: {t.numel()}")       # 60 (3*4*5)
print(f"Requires grad: {t.requires_grad}")  # False

## 2.3 — ⚠️ Corner Case: Data Types (dtype)

PyTorch defaults to `float32`, but wrong dtypes cause silent bugs or crashes.

In [ ]:
# ⚠️ Integer tensors lose precision in division!
a = torch.tensor([1, 2, 3])       # int64 by default
print(f"dtype: {a.dtype}")         # torch.int64
print(f"a / 2 = {a / 2}")         # Becomes float64!

# ⚠️ Mixing dtypes
b = torch.tensor([1.0, 2.0, 3.0])  # float32
# a + b works — auto-promotes int64 → float64

# ✅ Best practice: always specify dtype explicitly for ML
x = torch.tensor([1, 2, 3], dtype=torch.float32)
print(f"Explicit dtype: {x.dtype}")

# Type casting
x_int = x.int()           # or x.to(torch.int32)
x_double = x.double()     # or x.to(torch.float64)
x_half = x.half()         # float16 — for mixed precision training

# ⚠️ Corner Case: float16 overflow
big = torch.tensor([65536.0], dtype=torch.float16)
print(f"float16 overflow: {big}")  # inf! float16 max is ~65504

## 2.4 — Indexing and Slicing

In [ ]:
t = torch.arange(20).reshape(4, 5)
print("Original:\n", t)

# Basic indexing
print(f"Row 0:      {t[0]}")
print(f"Element[1,3]: {t[1, 3]}")
print(f"Last row:   {t[-1]}")

# Slicing
print(f"Rows 1-2:   {t[1:3]}")
print(f"Column 2:   {t[:, 2]}")
print(f"Submatrix:\n{t[1:3, 2:4]}")

# Boolean indexing (masking)
mask = t > 10
print(f"Elements > 10: {t[mask]}")

# Fancy indexing — select specific indices
rows = torch.tensor([0, 2, 3])
print(f"Rows [0,2,3]:\n{t[rows]}")

## 2.5 — Reshaping Operations

> ⚠️ **Critical Corner Case**: `view()` vs `reshape()` vs `contiguous()`

In [ ]:
t = torch.arange(12)

# reshape — always works (may copy if not contiguous)
r = t.reshape(3, 4)
print("Reshaped:\n", r)

# view — ONLY works on contiguous tensors (no copy, shares memory)
v = t.view(3, 4)

# ⚠️ Corner Case: transpose makes tensors non-contiguous!
t2 = torch.arange(12).reshape(3, 4)
t2_t = t2.t()  # Transpose
print(f"Is transposed contiguous? {t2_t.is_contiguous()}")  # False!

# t2_t.view(12)  # ❌ RuntimeError! Not contiguous
fixed = t2_t.contiguous().view(12)  # ✅ Make contiguous first
# OR just use reshape() — it handles this automatically
fixed2 = t2_t.reshape(12)  # ✅ Always works

print(f"Fixed: {fixed}")

In [ ]:
# More reshaping operations
t = torch.randn(2, 3, 4)

# Flatten
flat = t.flatten()                   # → (24,)
flat_partial = t.flatten(1)          # Flatten dims 1+ → (2, 12)
print(f"Flatten all: {flat.shape}, Flatten from dim 1: {flat_partial.shape}")

# Squeeze (remove dimensions of size 1)
t2 = torch.randn(1, 3, 1, 4)
squeezed = t2.squeeze()              # → (3, 4)
squeeze_dim = t2.squeeze(0)          # Only squeeze dim 0 → (3, 1, 4)
print(f"Squeeze all: {squeezed.shape}, Squeeze dim 0: {squeeze_dim.shape}")

# Unsqueeze (add dimension of size 1)
t3 = torch.randn(3, 4)
unsqueezed = t3.unsqueeze(0)         # → (1, 3, 4)  — add batch dim
print(f"Unsqueeze: {unsqueezed.shape}")

# Expand and Repeat
t4 = torch.tensor([[1], [2], [3]])   # (3, 1)
expanded = t4.expand(3, 4)           # → (3, 4) — no memory copy!
repeated = t4.repeat(1, 4)           # → (3, 4) — copies memory
print(f"Expand:\n{expanded}")
print(f"Repeat:\n{repeated}")

## 2.6 — Stacking and Concatenation

In [ ]:
a = torch.tensor([[1, 2], [3, 4]])
b = torch.tensor([[5, 6], [7, 8]])

# cat — join along existing dimension
cat_rows = torch.cat([a, b], dim=0)   # (4, 2) — stack vertically
cat_cols = torch.cat([a, b], dim=1)   # (2, 4) — stack horizontally
print(f"cat dim=0:\n{cat_rows}")
print(f"cat dim=1:\n{cat_cols}")

# stack — join along NEW dimension
stacked = torch.stack([a, b], dim=0)  # (2, 2, 2) — creates new dim
print(f"stack:\n{stacked}")
print(f"stack shape: {stacked.shape}")

# ⚠️ Corner Case: cat vs stack
# cat: tensors must match on all dims EXCEPT the cat dim
# stack: tensors must have IDENTICAL shapes

## 2.7 — ⚠️ Corner Case: Views Share Memory!

In [ ]:
original = torch.tensor([1, 2, 3, 4, 5])

# view creates a VIEW — shares same underlying data!
view = original.view(1, 5)
view[0, 0] = 99

print(f"Original after modifying view: {original}")
# [99, 2, 3, 4, 5] — CHANGED! They share memory!

# ✅ Use .clone() for an independent copy
original2 = torch.tensor([1, 2, 3, 4, 5])
independent = original2.clone().view(1, 5)
independent[0, 0] = 99
print(f"Original after modifying clone: {original2}")
# [1, 2, 3, 4, 5] — Unchanged!

## ✏️ Exercises

In [ ]:
# Exercise 1: Create a 5x5 identity matrix, then extract the diagonal as a 1D tensor
# YOUR CODE HERE


# Exercise 2: Create a (3, 4) tensor of random integers 0-9.
# Replace all values > 5 with -1 using boolean masking.
# YOUR CODE HERE


# Exercise 3: Create two tensors of shape (2, 3). Stack them to get shape (2, 2, 3),
# then reshape to (4, 3). Verify the result makes sense.
# YOUR CODE HERE
